# Prediction Data Normalization

将原始预测数据（Proby xlsx / Tox21 csv）转换为统一的 parquet 格式，供 MyLabData 应用读取。

## 目录约定（可自由修改）

| 类型 | 路径 |
| :--- | :--- |
| 原始数据 | `RAW_PREDICTION_DIR / {Model} / {BatchCode}/*.xlsx / *.csv` |
| 归一化数据 | `NORMALIZED_PREDICTION_DIR / {Model} / {BatchCode}.parquet` |

## 使用流程

1. 将原始预测文件放入对应目录
2. 运行 Cell 2（导入 + 路径配置）并按需修改路径变量
3. 运行 Cell 3（扫描状态）查看哪些 batch 需要归一化
4. 按需运行 Proby 或 Tox21 的归一化 cell
5. 运行最后的校验 cell 确认结果
6. 回到应用点击 **Refresh** 按钮同步

In [ ]:
import os
import re
from pathlib import Path

import pandas as pd
import numpy as np

# ── 路径配置（按需修改） ──
# 原始预测文件目录：按 model / batch 存放 xlsx 或 csv
RAW_PREDICTION_DIR = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction")

# 归一化 parquet 输出目录：按 model 输出 {BatchCode}.parquet
NORMALIZED_PREDICTION_DIR = Path(r"C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction")

SUPPORTING_DIR = Path(r"C:\Users\Cenking\Documents\SwissTools\MyLabData\Supporting")

def model_raw_dir(model: str) -> Path:
    return RAW_PREDICTION_DIR / model

def model_norm_dir(model: str) -> Path:
    return NORMALIZED_PREDICTION_DIR / model

# ── 模型定义 ──
ACTIVE_MODELS = ["Proby", "Tox21"]

PROBY_RESULT_COLUMNS = [
    "abs", "emi", "plqy", "e", "log10e", "lifetime",
    "abs_fwhm_cm", "emi_fwhm_cm", "abs_fwhm_nm", "emi_fwhm_nm",
]

# raw xlsx 列名 → 归一化列名
_PROBY_COL_RENAME = {
    "abs fwhm (cm-1)": "abs_fwhm_cm",
    "emi fwhm (cm-1)": "emi_fwhm_cm",
    "abs fwhm (nm)": "abs_fwhm_nm",
    "emi fwhm (nm)": "emi_fwhm_nm",
}

# 原始 Proby xlsx 需要保留的列名（rename 前）
_PROBY_RAW_KEEP = [
    "smiles", "abs", "emi", "plqy", "e", "log10e", "lifetime",
    "abs fwhm (cm-1)", "emi fwhm (cm-1)", "abs fwhm (nm)", "emi fwhm (nm)",
]

TOX21_RESULT_COLUMNS = [
    "NR-AR", "NR-AR-LBD", "NR-AhR", "NR-Aromatase",
    "NR-ER", "NR-ER-LBD", "NR-PPAR-gamma",
    "SR-ARE", "SR-ATAD5", "SR-HSE", "SR-MMP", "SR-p53",
]

# ── 溶剂映射 ──
def load_solvent_map():
    md_path = SUPPORTING_DIR / "Solvent.md"
    mapping = {}
    if md_path.exists():
        for line in md_path.read_text(encoding="utf-8").splitlines():
            m = re.match(r"\|\s*`([^`]+)`\s*\|\s*\*\*([^*]+)\*\*\s*\|", line)
            if m:
                mapping[m.group(1)] = m.group(2)
    return mapping

def _parse_sheet_solvent(sheet_name):
    """从 Proby sheet 名（如 'CS(C)=O (8)'）提取溶剂 SMILES。"""
    m = re.match(r"^(.+?)\s*\(\d+\)$", sheet_name)
    return m.group(1) if m else sheet_name

solvent_map = load_solvent_map()
print(f"已加载 {len(solvent_map)} 个溶剂映射")
print(f"原始数据目录: {RAW_PREDICTION_DIR}")
print(f"归一化输出目录: {NORMALIZED_PREDICTION_DIR}")

已加载 26 个溶剂映射
原始数据目录: C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction
归一化输出目录: C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction


In [2]:
# ── 扫描所有 batch 状态（原始目录 + 归一化目录） ──

import pyarrow.parquet as pq

print("="*60)
print("  Prediction Data Status")
print("="*60)

for model in ACTIVE_MODELS:
    raw_model_dir = model_raw_dir(model)
    norm_model_dir = model_norm_dir(model)

    print(f"\n── {model} ──")
    print(f"  原始目录: {raw_model_dir}")
    print(f"  归一化目录: {norm_model_dir}")

    raw_batches = []
    if raw_model_dir.exists():
        raw_batches = sorted([
            d.name for d in raw_model_dir.iterdir()
            if d.is_dir() and not d.name.startswith("_")
        ])

    norm_batches = []
    if norm_model_dir.exists():
        norm_batches = sorted([p.stem for p in norm_model_dir.glob("*.parquet")])

    batches = sorted(set(raw_batches) | set(norm_batches))

    if not batches:
        print("  (无 batch 数据)")
        continue

    for bc in batches:
        raw_dir = raw_model_dir / bc
        norm_path = norm_model_dir / f"{bc}.parquet"

        if model == "Proby":
            raw_count = len(list(raw_dir.glob("*.xlsx"))) if raw_dir.exists() else 0
            raw_label = f"{raw_count} xlsx"
        else:
            raw_count = len(list(raw_dir.glob("*.csv"))) if raw_dir.exists() else 0
            raw_label = f"{raw_count} csv"

        if norm_path.exists():
            meta = pq.read_metadata(norm_path)
            status = f"✓ 已归一化 ({meta.num_rows} rows)"
        else:
            status = "✗ 未归一化"

        print(f"  {bc}: {raw_label} → {status}")

  Prediction Data Status

── Proby ──
  原始目录: C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Proby
  归一化目录: C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Proby
  BatchE001: 9 xlsx → ✓ 已归一化 (66011478 rows)
  BatchE002: 44 xlsx → ✓ 已归一化 (110608836 rows)
  BatchE003: 50 xlsx → ✓ 已归一化 (12999922 rows)
  BatchE004: 5 xlsx → ✓ 已归一化 (12999896 rows)
  BatchE005: 50 xlsx → ✓ 已归一化 (12999896 rows)
  BatchE006: 7 xlsx → ✓ 已归一化 (1642472 rows)
  BatchE007: 62 xlsx → ✓ 已归一化 (16117244 rows)
  BatchE008: 31 xlsx → ✗ 未归一化
  BatchE009: 25 xlsx → ✗ 未归一化
  BatchE010: 72 xlsx → ✗ 未归一化
  BatchE011: 72 xlsx → ✗ 未归一化
  BatchE012: 33 xlsx → ✗ 未归一化
  BatchE013: 8 xlsx → ✗ 未归一化
  BatchE014: 70 xlsx → ✗ 未归一化
  BatchE015: 7 xlsx → ✗ 未归一化
  BatchE016: 4 xlsx → ✗ 未归一化
  BatchE017: 6 xlsx → ✗ 未归一化
  BatchE018: 6 xlsx → ✗ 未归一化
  BatchE019: 48 xlsx → ✗ 未归一化
  BatchE020: 60 xlsx → ✗ 未归一化
  BatchE021: 2 xlsx → ✗ 未归一化
  BatchE022: 1 xlsx → ✗ 未归一化
  BatchE023: 1 xlsx → ✗ 未归一化
  BatchE024: 4 xlsx 

## Proby 归一化（流式 + Sheet 级检查点）

从 `RAW_PREDICTION_DIR/Proby/{batch_code}/` 读取所有 xlsx 文件，每个 sheet 对应一种溶剂，
合并为 long-table 格式（SMILES + Solvent + 10 指标列），
输出到 `NORMALIZED_PREDICTION_DIR/Proby/{batch_code}.parquet`。

**特性：**
- **自动扫描**：自动发现所有未归一化的 batch 并依次处理
- **流式读取**：使用 openpyxl read_only 模式逐行读取，峰值内存仅约 1 个 sheet 的大小
- **Sheet 级检查点**：每个 sheet 处理完立即写入 checkpoint parquet，中断后可从上次的 sheet 继续
- **单线程顺序处理**：避免多线程内存翻倍，openpyxl 的 XML 解析受 GIL 限制，多线程提速有限

**直接运行即可，无需手动设置 batch_code。**

In [4]:
# ━━━ 自动扫描未归一化的 Proby batch ━━━

from tqdm.notebook import tqdm
from openpyxl import load_workbook
import pyarrow as pa
import pyarrow.parquet as pq
import shutil
import json

raw_model_dir = model_raw_dir("Proby")
norm_model_dir = model_norm_dir("Proby")

# 找出所有有原始 xlsx 但没有对应 parquet 的 batch
all_raw_batches = sorted([
    d.name for d in raw_model_dir.iterdir()
    if d.is_dir() and not d.name.startswith("_") and list(d.glob("*.xlsx"))
]) if raw_model_dir.exists() else []

pending_batches = [
    bc for bc in all_raw_batches
    if not (norm_model_dir / f"{bc}.parquet").exists()
]

if not pending_batches:
    print("✓ 所有 Proby batch 均已归一化，无需处理。")
else:
    print(f"发现 {len(pending_batches)} 个未归一化的 Proby batch:")
    for bc in pending_batches:
        xlsx_count = len(list((raw_model_dir / bc).glob("*.xlsx")))
        print(f"  {bc}: {xlsx_count} xlsx")
    print()

# ── 归一化列名映射 ──
_keep_lower = {c.lower(): c for c in _PROBY_RAW_KEEP}
_rename_final = {"smiles": "SMILES"}
_rename_final.update(_PROBY_COL_RENAME)

# ── 逐 batch 处理 ──
for batch_idx, batch_code in enumerate(pending_batches):
    print(f"\n{'='*60}")
    print(f"  [{batch_idx+1}/{len(pending_batches)}] 正在处理 {batch_code}")
    print(f"{'='*60}")

    raw_dir = raw_model_dir / batch_code
    out_dir = norm_model_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{batch_code}.parquet"

    xlsx_files = sorted(raw_dir.glob("*.xlsx"))
    print(f"找到 {len(xlsx_files)} 个 xlsx 文件", flush=True)

    # ── 检查点目录 ──
    checkpoint_root = out_dir / "_checkpoints"
    checkpoint_dir = checkpoint_root / batch_code
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    done_ckpts = {p.stem for p in checkpoint_dir.glob("*.parquet")}

    # ── 构建 sheet 级任务列表 ──
    tasks = []
    for xlsx_path in xlsx_files:
        wb = load_workbook(xlsx_path, read_only=True)
        for si, sname in enumerate(wb.sheetnames):
            ckpt_key = f"{xlsx_path.stem}_sheet{si:02d}"
            tasks.append((xlsx_path, si, sname, ckpt_key))
        wb.close()

    todo_tasks = [t for t in tasks if t[3] not in done_ckpts]
    skipped = len(tasks) - len(todo_tasks)

    print(f"总 sheet 数: {len(tasks)} ({len(xlsx_files)} 文件 × sheets)", flush=True)
    if skipped > 0:
        print(f"✓ 已有 {skipped} 个 sheet 检查点，跳过；剩余 {len(todo_tasks)} 个待处理", flush=True)
    else:
        print(f"无已有检查点，将处理全部 {len(todo_tasks)} 个 sheet", flush=True)

    # ── 阶段 1: 逐 sheet 流式处理 ──
    if todo_tasks:
        sheet_bar = tqdm(total=len(todo_tasks), desc=f"{batch_code} Sheet 进度", unit="sheet", dynamic_ncols=True)
        mol_bar = tqdm(desc="分子处理进度", unit="mol", dynamic_ncols=True)
        processed_molecules = 0

        for xlsx_path, si, sname, ckpt_key in todo_tasks:
            solvent_smiles = _parse_sheet_solvent(sname)

            wb = load_workbook(xlsx_path, read_only=True)
            ws = wb[sname]

            rows_iter = ws.iter_rows(values_only=True)
            raw_header = [str(c).strip() if c else "" for c in next(rows_iter)]

            header_lower = [h.lower() for h in raw_header]
            keep_indices = []
            keep_raw_names = []
            for idx, h in enumerate(header_lower):
                if h in _keep_lower:
                    keep_indices.append(idx)
                    keep_raw_names.append(_keep_lower[h])

            if "smiles" not in [n.lower() for n in keep_raw_names]:
                wb.close()
                pd.DataFrame().to_parquet(checkpoint_dir / f"{ckpt_key}.parquet", index=False)
                sheet_bar.update(1)
                continue

            data = []
            for row in rows_iter:
                data.append(tuple(row[i] for i in keep_indices))

            wb.close()

            df = pd.DataFrame(data, columns=keep_raw_names)
            df = df.rename(columns=_rename_final)
            df["Solvent"] = solvent_smiles

            for col in PROBY_RESULT_COLUMNS:
                if col in df.columns:
                    df[col] = pd.to_numeric(df[col], errors="coerce")

            mol_count = len(df)

            df.to_parquet(checkpoint_dir / f"{ckpt_key}.parquet", index=False)
            del df, data

            processed_molecules += mol_count
            sheet_bar.update(1)
            sheet_bar.set_postfix(file=xlsx_path.stem, sheet=sname)
            mol_bar.update(mol_count)
            mol_bar.set_postfix(total=processed_molecules)

        sheet_bar.close()
        mol_bar.close()
    else:
        print("所有 sheet 已在之前的运行中完成，直接进入合并阶段", flush=True)

    # ── 阶段 2: 合并检查点 ──
    print("\n正在合并检查点文件...", flush=True)

    ckpt_files = sorted(checkpoint_dir.glob("*.parquet"))
    if not ckpt_files:
        print(f"⚠ {batch_code} 没有检查点文件可合并，跳过")
        continue

    writer = None
    total_rows_written = 0
    solvent_counts: dict[str, int] = {}

    try:
        for ckpt in tqdm(ckpt_files, desc=f"{batch_code} 合并进度", unit="file"):
            table = pq.read_table(ckpt)
            if table.num_rows == 0:
                del table
                continue
            if writer is None:
                writer = pq.ParquetWriter(str(out_path), table.schema)
            writer.write_table(table)

            n = table.num_rows
            total_rows_written += n

            if "Solvent" in table.column_names:
                sol_arr = table.column("Solvent").to_pylist()
                for s in set(sol_arr):
                    solvent_counts[s] = solvent_counts.get(s, 0) + sol_arr.count(s)

            del table
    finally:
        if writer is not None:
            writer.close()

    if total_rows_written == 0:
        print(f"⚠ {batch_code} 所有 xlsx 文件读取失败，跳过")
        continue

    # ── 清理检查点 ──
    shutil.rmtree(checkpoint_dir)
    if checkpoint_root.exists() and not any(checkpoint_root.iterdir()):
        checkpoint_root.rmdir()

    solvents = sorted(solvent_counts.keys())
    print(f"\n✓ {batch_code} 写入: {out_path}", flush=True)
    print(f"  总行数: {total_rows_written}", flush=True)
    print(f"  溶剂数: {len(solvents)}", flush=True)
    for s in solvents:
        label = solvent_map.get(s, s)
        print(f"    {label} ({s}): {solvent_counts[s]} rows", flush=True)

if pending_batches:
    print(f"\n{'='*60}")
    print(f"  全部完成！共处理 {len(pending_batches)} 个 batch")
    print(f"{'='*60}")

发现 9 个未归一化的 Proby batch:
  BatchE016: 4 xlsx
  BatchE017: 6 xlsx
  BatchE018: 6 xlsx
  BatchE019: 48 xlsx
  BatchE020: 60 xlsx
  BatchE021: 2 xlsx
  BatchE022: 1 xlsx
  BatchE023: 1 xlsx
  BatchE024: 4 xlsx


  [1/9] 正在处理 BatchE016
找到 4 个 xlsx 文件
总 sheet 数: 104 (4 文件 × sheets)
无已有检查点，将处理全部 104 个 sheet


BatchE016 Sheet 进度:   0%|          | 0/104 [00:00<?, ?sheet/s]

分子处理进度: 0mol [00:00, ?mol/s]


正在合并检查点文件...


BatchE016 合并进度:   0%|          | 0/104 [00:00<?, ?file/s]

...

BatchE019 合并进度:   0%|          | 0/1248 [00:00<?, ?file/s]


✓ BatchE019 写入: C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Proby\BatchE019.parquet
  总行数: 12460812
  溶剂数: 26
    Cy (C1CCCCC1): 479262 rows
    THF (C1CCOC1): 479262 rows
    Dioxane (C1COCCO1): 479262 rows
    MeCN (CC#N): 479262 rows
    Acetone (CC(C)=O): 479262 rows
    IPA (CC(C)O): 479262 rows
    Hex (CCCCCC): 479262 rows
    Heptane (CCCCCCC): 479262 rows
    n-BuOH (CCCCO): 479262 rows
    Bu2O (CCCCOCCCC): 479262 rows
    n-PrOH (CCCO): 479262 rows
    EtOH (CCO): 479262 rows
    EA (CCOC(C)=O): 479262 rows
    Et2O (CCOCC): 479262 rows
    DMF (CN(C)C=O): 479262 rows
    MeOH (CO): 479262 rows
    DMSO (CS(C)=O): 479262 rows
    PhMe (Cc1ccccc1): 479262 rows
    CCl4 (ClC(Cl)(Cl)Cl): 479262 rows
    CHCl3 (ClC(Cl)Cl): 479262 rows
    DCM (ClCCl): 479262 rows
    H2O (O): 479262 rows
    TFA (O=C(O)C(F)(F)F): 479262 rows
    TFE (OCC(F)(F)F): 479262 rows
    c1ccccc1 (c1ccccc1): 479262 rows
    c1ccncc1 (c1ccncc1): 479262 rows

  [5/9] 正在处理 BatchE020
找到 60 

BatchE020 Sheet 进度:   0%|          | 0/1560 [00:00<?, ?sheet/s]

分子处理进度: 0mol [00:00, ?mol/s]

KeyboardInterrupt: 

## Tox21 归一化

从 `RAW_PREDICTION_DIR/Tox21/{batch_code}/` 读取所有 csv 文件，保留 SMILES + 12 毒性指标列，
输出到 `NORMALIZED_PREDICTION_DIR/Tox21/{batch_code}.parquet`。

**修改下方 `batch_code` 变量后运行即可。**

In [23]:
# ━━━ 设置要归一化的 batch ━━━
batch_code = "BatchE024"  # ← 修改此处
cpu_cores = 8              # ← 使用核数；None 表示自动使用 os.cpu_count()
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━

from tqdm.notebook import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

raw_dir = model_raw_dir("Tox21") / batch_code
assert raw_dir.exists(), f"目录不存在: {raw_dir}"

csv_files = sorted(raw_dir.glob("*.csv"))
print(f"找到 {len(csv_files)} 个 csv 文件", flush=True)

if not csv_files:
    raise ValueError(f"{raw_dir} 下没有找到 csv 文件")

max_workers = cpu_cores if cpu_cores is not None else (os.cpu_count() or 1)
max_workers = max(1, min(max_workers, len(csv_files)))
print(f"使用并发 worker 数: {max_workers}", flush=True)


def _load_tox21_csv(csv_path: str):
    path_obj = Path(csv_path)
    df_local = pd.read_csv(path_obj)
    keep_cols = ["SMILES"] + [c for c in TOX21_RESULT_COLUMNS if c in df_local.columns]
    df_local = df_local[keep_cols]
    return path_obj.name, df_local, len(df_local), len(keep_cols) - 1


frames = []
processed_molecules = 0

file_bar = tqdm(total=len(csv_files), desc="Tox21 文件进度", unit="file", dynamic_ncols=True)
molecule_bar = tqdm(desc="分子处理进度", unit="mol", dynamic_ncols=True)

with ThreadPoolExecutor(max_workers=max_workers) as ex:
    future_map = {
        ex.submit(_load_tox21_csv, str(csv_path)): csv_path
        for csv_path in csv_files
    }

    for fut in as_completed(future_map):
        src = future_map[fut]
        try:
            filename, df, mol_count, metric_count = fut.result()
            frames.append(df)
            processed_molecules += mol_count

            file_bar.update(1)
            file_bar.set_postfix(file=filename)
            molecule_bar.update(mol_count)
            molecule_bar.set_postfix(total=processed_molecules)

            tqdm.write(f"完成 {filename}: {mol_count} rows, {metric_count} metrics")
        except Exception as e:
            file_bar.update(1)
            tqdm.write(f"读取失败 {src.name}: {e}")

file_bar.close()
molecule_bar.close()

if not frames:
    raise ValueError("所有 csv 文件读取失败，无法生成归一化结果")

combined = pd.concat(frames, ignore_index=True)
for col in TOX21_RESULT_COLUMNS:
    if col in combined.columns:
        combined[col] = pd.to_numeric(combined[col], errors="coerce")

out_dir = model_norm_dir("Tox21")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"{batch_code}.parquet"
combined.to_parquet(out_path, index=False)

print(f"\n✓ 写入: {out_path}", flush=True)
print(f"  总行数: {len(combined)}", flush=True)
print(f"  列: {list(combined.columns)}", flush=True)

找到 4 个 csv 文件
使用并发 worker 数: 4


Tox21 文件进度:   0%|          | 0/4 [00:00<?, ?file/s]

分子处理进度: 0mol [00:00, ?mol/s]

完成 BatchE024Slice04.csv: 4887 rows, 12 metrics
完成 BatchE024Slice02.csv: 10000 rows, 12 metrics
完成 BatchE024Slice03.csv: 10000 rows, 12 metrics
完成 BatchE024Slice01.csv: 10000 rows, 12 metrics

✓ 写入: C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Tox21\BatchE024.parquet
  总行数: 34887
  列: ['SMILES', 'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']


## 校验

读取指定的归一化 parquet，显示基本信息和前几行数据。

In [24]:
# ━━━ 选择要校验的 model 和 batch ━━━
model = "Tox21"       # ← "Proby" 或 "Tox21"
batch_code = "BatchE010"  # ← 修改此处
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

pq_path = model_norm_dir(model) / f"{batch_code}.parquet"
assert pq_path.exists(), f"文件不存在: {pq_path}"

df = pd.read_parquet(pq_path)
print(f"文件: {pq_path}")
print(f"行数: {len(df)}")
print(f"列名: {list(df.columns)}")
print()

if model == "Proby" and "Solvent" in df.columns:
    solvents = sorted(df["Solvent"].unique())
    print(f"溶剂数: {len(solvents)}")
    for s in solvents:
        label = solvent_map.get(s, s)
        print(f"  {label}: {len(df[df['Solvent']==s])} rows")
    print()

print("── 数值统计 ──")
result_cols = PROBY_RESULT_COLUMNS if model == "Proby" else TOX21_RESULT_COLUMNS
cols_in_df = [c for c in result_cols if c in df.columns]
display(df[cols_in_df].describe().round(4))

print("\n── 前 5 行 ──")
display(df.head())

文件: C:\Users\Cenking\Documents\ExperimentData\DryData\Prediction\Tox21\BatchE010.parquet
行数: 719995
列名: ['SMILES', 'NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']

── 数值统计 ──


,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53
count,719995.0000,719995.0000,719995.0000,719995.0000,719995.0000,719995.0000,719995.0000,719995.0000,719995.0000,719995.0000,719995.0000,719995.0000
mean,0.0323,0.0175,0.2317,0.0655,0.1038,0.0295,0.0325,0.1783,0.0537,0.0505,0.1339,0.0858
std,0.0260,0.0264,0.2374,0.0784,0.0721,0.0346,0.0500,0.1497,0.0765,0.0681,0.1818,0.1111
min,0.0000,0.0000,0.0000,0.0000,0.0004,0.0000,0.0000,0.0005,0.0000,0.0000,0.0000,0.0000
25%,0.0143,0.0044,0.0406,0.0130,0.0552,0.0099,0.0036,0.0691,0.0071,0.0094,0.0139,0.0118
50%,0.0250,0.0094,0.1330,0.0341,0.0839,0.0177,0.0129,0.1261,0.0220,0.0254,0.0484,0.0368
75%,0.0420,0.0192,0.3712,0.0881,0.1285,0.0342,0.0388,0.2430,0.0643,0.0620,0.1789,0.1156
max,0.3208,0.6107,0.9526,0.8493,0.9656,0.9224,0.7263,0.9233,0.7003,0.9050,0.9978,0.7956



── 前 5 行 ──


,SMILES,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53
0,Brc1cccc(N2CCCCC2)c1,0.022998,0.004647,0.140987,0.058688,0.144234,0.028261,0.002019,0.212250,0.006477,0.015930,0.116256,0.005299
1,CNCc1nc(-c2cccs2)no1,0.011278,0.003371,0.329524,0.019173,0.052040,0.011800,0.004245,0.122580,0.033106,0.020759,0.024052,0.097812
2,NCCc1nc2cc(Cl)ccc2[nH]1,0.026653,0.007914,0.732851,0.058596,0.057804,0.011872,0.008155,0.187782,0.035770,0.061294,0.118142,0.063327
3,COC(=O)CC(N)c1ccccc1,0.012458,0.005172,0.026852,0.008952,0.057276,0.004524,0.003610,0.019335,0.006206,0.005533,0.011066,0.003018
4,Cc1cccc(N2CCCC2=N)c1,0.044305,0.009953,0.098720,0.022045,0.102794,0.014442,0.005815,0.062249,0.009579,0.019128,0.015929,0.003617
